# Multi-Agent Systems with LangGraph
_Kapruka Gift Concierge - Declarative Graph-Based Orchestration_

---

This notebook shows the current Kapruka LangGraph orchestrator: a `StateGraph` with specialist agents, shared state, and multi-route fan-out.

## Why LangGraph?

| Earlier orchestration limitation | Current LangGraph approach |
|---|---|
| Routing lived inside imperative Python control flow | Routing is compiled into graph edges |
| One assistant handled every route | Specialist nodes handle profile, catalog, and concierge work |
| Compound questions had to be serialized manually | Parallel fan-out plus `merge_responses` combines results |
| Debugging meant reading logs | `AgentState`, Mermaid graph views, and per-node traces make execution explicit |

## Kapruka graph topology

```text
recall -> supervisor -> [profile_agent | catalog_agent | concierge_agent]
                       -> merge_responses -> save_memory -> END
```

### Route mapping in the current project

| Router output | Specialist node | Responsibility |
|---|---|---|
| `crm` | `profile_agent_node` | User profile lookups and structured delivery/logistics checks |
| `rag` | `catalog_agent_node` | Product catalog retrieval, internal FAQ, recommendation context |
| `web_search` | `concierge_agent_node` | Live external facts such as weather or traffic disruptions |
| `direct` | `concierge_agent_node` | Greetings, memory-only turns, and plain concierge replies |

### New capability: multi-route fan-out

The current Kapruka orchestrator can return more than one route for a compound request. Example:

- "Recommend a birthday gift under Rs. 6000 and also check same-day delivery in Kandy"
- Router output: `[{route: rag}, {route: crm, action: check_delivery_coverage}]`
- LangGraph runs both specialist nodes, then `merge_responses` produces one customer-facing answer.


---
## Section 0 - Setup


In [ ]:
import sys, os, re
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv()

from infrastructure.log import setup_logging
from loguru import logger
setup_logging("INFO", for_notebook=True)

import pandas as pd
from sqlalchemy.orm import sessionmaker
from infrastructure.db import get_sql_engine
from infrastructure.db.crm_models import (
    User,
    DeliveryZone,
    DeliverySlot,
    CourierProfile,
    ProductDeliveryRule,
)

logger.success("Environment ready")


---
## Section 1 - Build and Visualize the Graph

One of the main advantages of LangGraph is that the orchestrator is literally a graph. For Kapruka, that means we can inspect the exact routing topology instead of reverse-engineering control flow from Python methods.


In [ ]:
from agents.orchestrator import build_agent
from langchain_core.messages import HumanMessage

agent = build_agent(enable_crm=True, enable_rag=True, enable_web=True)

logger.success("LangGraph agent built")
logger.info(f"  CRM tool        : {'ON' if agent.crm_tool else 'OFF'}")
logger.info(f"  RAG tool        : {'ON' if agent.rag_tool else 'OFF'}")
logger.info(f"  Web search tool : {'ON' if agent.web_tool else 'OFF'}")


In [ ]:
# Visualize the StateGraph.
# draw_mermaid_png() renders via the Mermaid.ink API and returns PNG bytes.
# We wrap it in IPython.display.Image so Jupyter shows it inline.

from IPython.display import Image, display

graph_png = agent.graph.get_graph(xray=True).draw_mermaid_png()
display(Image(graph_png))

print("Raw Mermaid (paste into https://mermaid.live):")
print(agent.graph.get_graph(xray=True).draw_mermaid())


In [ ]:
# Inspect registered nodes and edges programmatically.
# Each edge is a NamedTuple: Edge(source, target, data, conditional)
g = agent.graph.get_graph()

print("Nodes:")
for node_id in g.nodes:
    print(f"  - {node_id}")

print("Edges:")
for edge in g.edges:
    kind = " [conditional]" if edge.conditional else ""
    label = f" ({edge.data})" if edge.data else ""
    print(f"  {edge.source:30s} -> {edge.target}{label}{kind}")


---
## Section 2 - AgentState: The Shared Conveyor Belt

In the earlier orchestration style, state was implicit and mostly passed through function arguments. In the current graph, every node reads from and writes back to the same `AgentState` TypedDict.

The important Kapruka-specific fields are:

- `memory_context`: short-term conversation text assembled by `recall_node`
- `semantic_facts`: structured long-term memory facts such as budgets, gift preferences, recipient details, and delivery habits
- `route_decision`: the primary route for backward compatibility
- `route_decisions`: all routes for multi-intent fan-out
- `agent_outputs`: reducer-backed list that collects parallel specialist results before merge

This makes the graph inspectable and also enables true multi-route execution without custom state-merging code.


In [ ]:
from agents.state import AgentState

print("AgentState fields:")
print("=" * 72)
hints = AgentState.__annotations__
for field, annotation in hints.items():
    print(f"  {field:20s} : {annotation}")


---
## Section 3 - Supervisor and Specialist Pattern

The current Kapruka graph is a supervisor-worker system:

```text
                 [supervisor_node]
                 QueryRouter.route()
                 route_decisions -> one or more specialist nodes
                      |                |                |
                      v                v                v
               [profile_agent]   [catalog_agent]   [concierge_agent]
```

### Specialist personas in the current project

| Node | Role | Tools | Typical queries |
|---|---|---|---|
| `profile_agent_node` | CRM and logistics specialist | `CRMTool` | profile lookup, delivery coverage, slots, courier search |
| `catalog_agent_node` | product and knowledge specialist | `RAGTool` | gift recommendations, catalog search, internal delivery FAQ |
| `concierge_agent_node` | direct concierge and live-web specialist | `WebSearchTool` when needed | greetings, memory-only turns, live traffic/weather/news |
| `merge_responses_node` | answer combiner for fan-out | merge synthesiser prompt | compound questions spanning multiple tool paths |

### Current route rules

- `crm` -> `profile_agent_node`
- `rag` -> `catalog_agent_node`
- `web_search` -> `concierge_agent_node`
- `direct` -> `concierge_agent_node`

Unlike the earlier draft, the current project splits by profile/logistics, catalog/knowledge, and concierge/live information.


---
## Section 4 - Demo: Concierge Agent

A greeting should route to `direct` and stay inside the concierge persona. A live disruption question should still use the concierge node, but with the `web_search` path enabled.


In [ ]:
def extract_phone(text: str) -> str:
    match = re.search(r"\+?[\d][\d\s\-\.\(\)]{7,18}[\d]", text)
    if not match:
        raise ValueError("No phone number found.")
    raw = re.sub(r"\D", "", match.group())
    if raw.startswith("0") and len(raw) == 10:
        raw = "94" + raw[1:]
    elif len(raw) == 9 and not raw.startswith("94"):
        raw = "94" + raw
    return raw

def _crm_session():
    return sessionmaker(bind=get_sql_engine())()

def resolve_user_id(external_user_id: str) -> str:
    session = _crm_session()
    try:
        user = session.query(User).filter(User.external_user_id == external_user_id).first()
        if not user:
            raise ValueError(
                f"No CRM user found for external_user_id={external_user_id}."
            )
        return user.user_id
    finally:
        session.close()

def show_user(external_user_id: str) -> None:
    session = _crm_session()
    try:
        user = session.query(User).filter(User.external_user_id == external_user_id).first()
        if not user:
            logger.warning(f"No CRM user found for external_user_id={external_user_id}")
            return

        rows = [
            {"Field": "User ID", "Value": user.user_id or "-"},
            {"Field": "External ID", "Value": user.external_user_id or "-"},
            {"Field": "Name", "Value": user.full_name or "-"},
            {"Field": "Phone", "Value": user.phone or "-"},
            {"Field": "Email", "Value": user.email or "-"},
            {"Field": "District", "Value": user.district or "-"},
            {"Field": "Province", "Value": user.province or "-"},
            {"Field": "Address", "Value": user.address or "-"},
            {"Field": "Notes", "Value": user.notes or "-"},
            {"Field": "Active", "Value": "Yes" if user.active else "No"},
        ]

        print("CRM User Record (Supabase users table)")
        display(pd.DataFrame(rows).style.hide(axis="index"))
    finally:
        session.close()
        
def show(resp, label=""):
    print("" + "=" * 72)
    if label:
        print(f"{label}")
        print("-" * 72)
    print(f"Primary route : {resp.route}")
    print(f"All routes    : {resp.routes}")
    print(f"Action        : {resp.action}")
    print(f"Latency (ms)  : {resp.latency_ms}")
    ctx_lines = len((resp.memory_context or "").splitlines()) if resp.memory_context else 0
    print(f"Memory lines  : {ctx_lines}")
    if resp.tool_output:
        preview = resp.tool_output[:500]
        print(f"Tool output   :{preview}{'...' if len(resp.tool_output) > 500 else ''}")
    print("Answer:")
    print(resp.answer)
    print("=" * 72)


In [ ]:
# The greeting carries the user's identity.
greeting = "Hi, I'm Thilanka. My mobile number is 077 123 4567."
EXTERNAL_USER_ID = extract_phone(greeting)
USER_ID = resolve_user_id(EXTERNAL_USER_ID)
SESSION_ID = "nb01-demo"

logger.success(f"EXTERNAL_USER_ID  = {EXTERNAL_USER_ID}")
logger.success(f"USER_ID           = {USER_ID}")
logger.info(f"SESSION  = {SESSION_ID}")
show_user(EXTERNAL_USER_ID)

resp = agent.chat(user_message=greeting, user_id=USER_ID, session_id=SESSION_ID)
show(resp, "CONCIERGE AGENT - direct greeting")

print("Expectation:")
print("  - Supervisor classifies the turn as direct")
print("  - concierge_agent_node answers without using a tool")

In [ ]:
# Live external information still routes through the concierge specialist.
resp_web = agent.chat(
    user_message="Is heavy rain or traffic affecting Colombo deliveries today?",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_web, "CONCIERGE AGENT - web_search fallback")

print("Note:")
print("  - Route should be web_search")
print("  - The specialist node is still concierge_agent_node")
print("  - The difference is that the node uses WebSearchTool before replying")


---
## Section 5 - Demo: Catalog Agent

The catalog specialist handles product search, internal delivery knowledge, and recommendation-style questions. It also gets the richest customer preference context from long-term memory.

We will first store preference facts, then ask a recommendation question that should route to `rag`.


In [ ]:
# Seed customer preference memory for later catalog retrieval.
resp_mem = agent.chat(
    user_message=(
        """Please remember this for future gift suggestions: I usually send gifts
        to my sister in Kandy, my budget is around Rs. 6000, she likes
        chocolates and flowers, and we should avoid peanut products."""
    ),
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_mem, "CONCIERGE AGENT - memory-building turn")

print("The memory distiller should extract facts about district, budget, preferences, and restrictions.")


In [ ]:
# Catalog / recommendation question -> catalog_agent_node via rag.
resp_rag = agent.chat(
    user_message="Recommend a birthday gift under Rs. 6000 for my sister in Kandy and keep the peanut restriction in mind.",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_rag, "CATALOG AGENT - RAG plus customer preferences")

print("Observe:")
print("  - Route should be rag")
print("  - catalog_agent_node uses RAGTool for internal product knowledge")
print("  - semantic_facts should provide budget, district, and allergy-like preference context")


In [ ]:
# Similar semantic intent -> CAG cache hit on the second pass when cache is populated.
resp_cached = agent.chat(
    user_message="What birthday gifts would fit that same Kandy delivery and Rs. 6000 budget?",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_cached, "CATALOG AGENT - RAG cache follow-up")

if agent.rag_tool:
    stats = agent.rag_tool.cache_stats()
    print("CAG cache stats:")
    for key, value in stats.items():
        print(f"  {key}: {value}")


---
## Section 6 - Demo: Profile Agent

The profile specialist covers two kinds of CRM-backed work in this project:

- stable customer profile lookups and updates
- structured delivery/logistics checks from SQL tables

These are table-driven requests, so they should route to `crm` and execute inside `profile_agent_node`.


In [ ]:
# Delivery feasibility question -> profile_agent_node, action=check_delivery_coverage.
resp_crm = agent.chat(
    user_message="Can you check same-day delivery availability in Kandy for a cake? My phone number is 077 123 4567.",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_crm, "PROFILE AGENT - CRM delivery coverage")

session = sessionmaker(bind=get_sql_engine())()
try:
    zone = session.query(DeliveryZone).filter(DeliveryZone.district == "Kandy").first()
    rule = session.query(ProductDeliveryRule).filter(ProductDeliveryRule.product_type == "cake").first()
    slots = (
        session.query(DeliverySlot)
        .filter(DeliverySlot.district == "Kandy")
        .order_by(DeliverySlot.slot.asc())
        .limit(5)
        .all()
    )

    if zone:
        display(pd.DataFrame([zone.to_dict()]))
    if rule:
        display(pd.DataFrame([rule.to_dict()]))
    if slots:
        display(pd.DataFrame([slot.to_dict() for slot in slots]))
finally:
    session.close()


In [ ]:
# Another CRM-style logistics request -> profile_agent_node, action=search_couriers.
resp_couriers = agent.chat(
    user_message="Which couriers are available in Colombo for deliveries today?",
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_couriers, "PROFILE AGENT - CRM courier lookup")

session = sessionmaker(bind=get_sql_engine())()
try:
    couriers = (
        session.query(CourierProfile)
        .filter(CourierProfile.district == "Colombo", CourierProfile.availability == True)
        .limit(5)
        .all()
    )
    if couriers:
        display(pd.DataFrame([c.to_dict() for c in couriers]))
finally:
    session.close()


---
## Section 7 - Structured Memory: `semantic_facts` as `list[dict]`

This is one of the key upgrades from the imperative version.

### Earlier pattern

```python
memory_context = recaller.format_context(st_turns)
# The single assistant received one flattened text blob.
```

### Current LangGraph pattern

```python
state['memory_context'] = recaller.format_context(st_turns)
state['semantic_facts'] = [fact.to_dict() for fact in lt_facts]
state['route_decisions'] = [...]      # for fan-out
state['agent_outputs'] = [...]        # reducer-backed merge input
```

In the Kapruka project, `semantic_facts` usually contain preference and logistics context such as:

- recipient relationship and district
- budget range
- preferred categories
- food restrictions or dislikes
- recurring delivery habits

The catalog specialist consumes this heavily for recommendation quality, while the profile specialist focuses more on structured SQL-backed facts.


In [ ]:
# Query the long-term store directly to inspect structured customer facts.
from infrastructure.llm import get_default_embeddings
from memory import LongTermMemoryStore
import json

embedder = get_default_embeddings()
lt_store = LongTermMemoryStore(embedder)

facts = lt_store.query(
    USER_ID,
    "gift preferences budget delivery district allergies dislikes",
    k=5,
    threshold=0.2,
)

print(f"LT facts for user {USER_ID}:")
print("-" * 72)
for i, fact in enumerate(facts, 1):
    print(f"Fact {i}:")
    print(f"  text  : {fact.text}")
    print(f"  tags  : {fact.tags}")
    print(f"  score : {fact.score:.3f}")
    print()
    
structured = [{"text": f.text, "tags": f.tags, "score": f.score} for f in facts]
print("Structured payload that can travel inside AgentState:")
print(json.dumps(structured[:3], indent=2))


---
## Section 8 - Full Multi-Turn Conversation

Now we run a realistic Kapruka conversation that exercises the three specialist behaviors across multiple turns:

1. concierge direct greeting
2. concierge memory-building turn
3. profile logistics lookup
4. catalog recommendation retrieval
5. concierge memory recall question
6. concierge web-search disruption check

The graph stays the same for every turn. Only the conditional edge choices change.


In [ ]:
FULL_SESSION = "nb04-full-demo"
first = "Hi, I'm Thilanka. My mobile number is 077 123 4567 and I usually send gifts to my sister in Kandy."
FULL_USER = extract_phone(first)

turns = [
    (first, "direct"),
    ("Please remember that my budget is usually Rs. 6000 and avoid peanut-based gifts.", "direct"),
    ("Can you check whether same-day delivery is available in Kandy for chocolates?", "crm"),
    ("Recommend a birthday gift for my sister in Kandy under Rs. 6000.", "rag"),
    ("What do you remember about my gift preferences and delivery needs?", "direct"),
    ("Is heavy rain or traffic affecting Colombo deliveries today?", "web_search"),
]

print("Full multi-turn Kapruka conversation")
print("=" * 72)

for i, (msg, expected_route) in enumerate(turns, 1):
    resp = agent.chat(user_message=msg, user_id=USER_ID, session_id=FULL_SESSION)
    ctx_lines = len(resp.memory_context.strip().split("")) if resp.memory_context.strip() else 0
    match = "OK" if resp.route == expected_route else f"MISMATCH expected={expected_route}"

    print(f"Turn {i}: {msg}")
    print(f"  route      : {resp.route} {('/ ' + resp.action) if resp.action else ''}")
    print(f"  routes     : {resp.routes}")
    print(f"  memory     : {ctx_lines} lines")
    print(f"  latency_ms : {resp.latency_ms}")
    print(f"  check      : {match}")
    preview = resp.answer[:220]
    print(f"  answer     : {preview}{'...' if len(resp.answer) > 220 else ''}")
    print("-" * 72)

logger.success("Conversation complete")


---
## Section 9 - State Inspection

LangGraph makes state inspection straightforward. We can invoke the graph directly and inspect exactly what moved through `AgentState`: the routing decision set, recalled memory, collected specialist outputs, and merged answer.


In [ ]:
# Run a compound query and inspect raw AgentState.
from langchain_core.messages import HumanMessage

INSPECT_SESSION = "nb04-inspect"
test_message = (
    "Recommend a birthday gift under Rs. 6000 for my sister in Kandy and also "
    "check whether same-day delivery is available there for chocolates."
)

initial_state = {
    "messages": [HumanMessage(content=test_message)],
    "user_id": FULL_USER,
    "session_id": INSPECT_SESSION,
    "agent_outputs": [],
}

final_state = agent.graph.invoke(initial_state)

print("=" * 72)
print("Final AgentState after graph traversal")
print("=" * 72)

print("[route_decision]")
print(final_state.get("route_decision"))

print("[route_decisions]")
for item in final_state.get("route_decisions", []) or []:
    print(f"  - {item}")

print("[agent_outputs]")
for output in final_state.get("agent_outputs", []) or []:
    print(f"  - route={output.get('route')} answer_preview={output.get('answer', '')[:120]}")

mc = final_state.get("memory_context") or ""
print("[memory_context preview]")
print(mc[:400])

facts = final_state.get("semantic_facts") or []
print(f"[semantic_facts] count={len(facts)}")
for fact in facts[:3]:
    print(f"  - {fact.get('text', '')} | tags={fact.get('tags', [])}")

print("[tool_output preview]")
print((final_state.get("tool_output") or "")[:500])

print("[final_answer preview]")
print((final_state.get("final_answer") or "")[:500])
print("=" * 72)


---
## Section 10 - Multi-Route: LangGraph Fan-Out

This is the biggest orchestration upgrade in the current project.

For compound questions:

1. `recall_node` loads memory once
2. `supervisor_node` returns more than one route decision
3. `supervisor_routing()` maps those decisions to multiple specialist nodes
4. LangGraph fans out into parallel branches
5. `merge_responses_node` combines specialist answers into one reply
6. `save_memory_node` stores the merged turn

For single-route questions, `merge_responses_node` becomes a pass-through and adds no extra merge call.


In [ ]:
# Multi-route demo: product recommendation plus logistics check.
from langchain_core.messages import HumanMessage

multi_message = (
    "Recommend a birthday gift under Rs. 6000 for my sister in Kandy and also "
    "check whether same-day delivery is available there for chocolates."
)

resp_multi = agent.chat(
    user_message=multi_message,
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_multi, "MULTI-ROUTE - RAG plus CRM fan-out")

raw_multi = agent.graph.invoke(
    {
        "messages": [HumanMessage(content=multi_message)],
        "user_id": USER_ID,
        "session_id": SESSION_ID + "-fanout",
        "agent_outputs": [],
    }
)

print("Route decisions returned by the supervisor:")
for item in raw_multi.get("route_decisions", []) or []:
    print(f"  - {item}")
print(f"Merged specialist outputs: {len(raw_multi.get('agent_outputs', []) or [])}")


In [ ]:
# Single-route query still follows the same graph, but no real fan-out occurs.
single_message = "Recommend a birthday gift under Rs. 5000 for someone who likes flowers."
resp_single = agent.chat(
    user_message=single_message,
    user_id=USER_ID,
    session_id=SESSION_ID,
)
show(resp_single, "SINGLE-ROUTE - merge is effectively a pass-through")

print("No merge synthesis is needed when there is only one specialist output.")
print(f"Routes taken: {resp_single.routes}")


---
## Section 11 - LangFuse: Per-Node Observability

With the graph-based orchestrator, LangFuse traces line up with node boundaries more cleanly:

```text
agent_chat
  recall_node
  supervisor_node
  profile_agent_node / catalog_agent_node / concierge_agent_node
  merge_responses_node   (only when multi-route)
  save_memory_node
```

Within those spans you should also see the tool-level work:

- `kapruka_rag_search` for the catalog specialist
- CRM dispatch inside the profile specialist
- `web_search` inside concierge when live data is needed
- distillation work inside `save_memory_node` when a turn should produce long-term facts

This makes it much easier to answer practical debugging questions:

- Which node dominated latency?
- Did the router produce one route or many?
- Did RAG hit the semantic cache?
- Was a merge pass required?
- What memory context was available before synthesis?


In [ ]:
from infrastructure.observability import flush

flush()

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"LangFuse dashboard: {host}")
print()
print("What to look for:")
print("  - Traces: one per .chat() call")
print("  - Spans : recall_node, supervisor_node, profile/catalog/concierge nodes")
print("  - Merge : merge_responses_node should appear only on fan-out queries")
print("  - Metadata: route_decisions, recalled facts, cache behavior, token usage")


---
## Section 12 - Orchestrator Comparison

| Aspect | Earlier imperative orchestrator | Current LangGraph orchestrator |
|---|---|---|
| Orchestration | Private Python control flow | Compiled `StateGraph` |
| State | Implicit function arguments | Explicit `AgentState` |
| Routing | One route at a time in `_dispatch()` | Conditional edges plus multi-route fan-out |
| Specialists | One assistant adapting by prompt | Profile, catalog, concierge specialists |
| Merge behavior | Manual serialization of tool work | `merge_responses_node` for compound requests |
| Memory payload | Mainly flattened display text | Display text plus structured `semantic_facts` |
| Debugging | Logs and ad-hoc prints | Graph visualization plus inspectable state and node traces |

### What stayed the same

The LangGraph upgrade is mostly an orchestration change. The core building blocks remain the same:

- memory stores and distillation
- CRM client and logistics tables
- RAG / CAG / CRAG services
- embeddings, LLM wiring, observability, and configuration
- Kapruka knowledge-base content in Qdrant

In other words, the main change is how work is coordinated, not the underlying Kapruka domain or supporting services.


---
## Summary

| Component | Role | Key file |
|---|---|---|
| `StateGraph` | compiled orchestration graph | `src/agents/orchestrator.py` |
| `AgentState` | shared state flowing through all nodes | `src/agents/state.py` |
| `recall_node` | loads short-term context and long-term facts | `src/agents/orchestrator.py` |
| `supervisor_node` | router invocation and route serialization | `src/agents/orchestrator.py` |
| `profile_agent_node` | CRM-backed user profile and logistics specialist | `src/agents/orchestrator.py` |
| `catalog_agent_node` | RAG-backed product and knowledge specialist | `src/agents/orchestrator.py` |
| `concierge_agent_node` | direct replies and live external search | `src/agents/orchestrator.py` |
| `merge_responses_node` | combines parallel specialist answers | `src/agents/orchestrator.py` |
| `save_memory_node` | persists the turn and optionally distills facts | `src/agents/orchestrator.py` |
| `CRMTool` | stable profile and logistics access | `src/agents/tools/crm_tool.py` |
| `RAGTool` | catalog retrieval plus semantic cache | `src/agents/tools/rag_tool.py` |
| `WebSearchTool` | live weather, traffic, and external updates | `src/agents/tools/web_search_tool.py` |
| LangFuse spans | per-node observability | `src/infrastructure/observability.py` |
